In [1]:
from sentinelhub import (
    SHConfig, SentinelHubRequest, DataCollection,
    MimeType, CRS, BBox
)
import os

In [9]:
config = SHConfig()
config.sh_client_id = 'sh-16f56f32-9a21-4587-99e7-3ccc42527d71'
config.sh_client_secret = 'JeS1lvmqH5B1kZTnE9AiBRKs138nkOWs'
config.sh_base_url = 'https://sh.dataspace.copernicus.eu'
config.sh_token_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'


print(config)

if not config.sh_client_id or not config.sh_client_secret:
    print("Warning! To use Process API, please provide the credentials (OAuth client ID and client secret).")

{
  "instance_id": "",
  "sh_client_id": "***********************************7d71",
  "sh_client_secret": "****************************kOWs",
  "sh_base_url": "https://sh.dataspace.copernicus.eu",
  "sh_auth_base_url": null,
  "sh_token_url": "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token",
  "geopedia_wms_url": "https://service.geopedia.world",
  "geopedia_rest_url": "https://www.geopedia.world/rest",
  "aws_access_key_id": "",
  "aws_secret_access_key": "",
  "aws_session_token": "",
  "aws_metadata_url": "https://roda.sentinel-hub.com",
  "aws_s3_l1c_bucket": "sentinel-s2-l1c",
  "aws_s3_l2a_bucket": "sentinel-s2-l2a",
  "opensearch_url": "http://opensearch.sentinel-hub.com/resto/api/collections/Sentinel2",
  "max_wfs_records_per_query": 100,
  "max_opensearch_records_per_query": 500,
  "max_download_attempts": 4,
  "download_sleep_time": 5.0,
  "download_timeout_seconds": 120.0,
  "number_of_download_processes": 1,
  "max_retries": null
}


In [3]:
bbox = BBox([18.51635, 48.78376, 18.80255, 49.04104], crs=CRS.WGS84)

evalscript = """//VERSION=3
function setup() {
  return {
    input: ["B04", "B08", "CLD"],
    output: {
      bands: 1,
      sampleType: "FLOAT32"
    }
  };
}

function evaluatePixel(sample) {
  let ndvi = (sample.B08 - sample.B04) / (sample.B08 + sample.B04);

  // cloud threshold (napr. 20%)
  if (sample.CLD > 20) {
    return [NaN];
  }

  return [ndvi];
}
"""

In [4]:
months = [
    ("2020-06-01", "2020-07-01"),
    # ("2020-07-01", "2020-08-01"),
    # ("2020-08-01", "2020-09-01"),
    # ("2020-09-01", "2020-10-01"),
    # ("2020-10-01", "2020-11-01"),
]

for start, end in months:
    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(start, end),
                mosaicking_order="leastCC"  # dôležité
            )
        ],
        responses=[
            SentinelHubRequest.output_response(
                "default", MimeType.TIFF
            )
        ],
        bbox=bbox,
        size=(512, 512),
        config=config
    )

    data = request.get_data()

    filename = f"ndvi_{start[:7]}.tif"
    with open(filename, "wb") as f:
        f.write(data[0])

    print(f"Saved {filename}")

DownloadFailedException: Failed to download from:
https://services.sentinel-hub.com/api/v1/process
with HTTPError:
401 Client Error: Unauthorized for url: https://services.sentinel-hub.com/api/v1/process
Server response: "{"status": 401, "reason": "Unauthorized", "message": "You are not authorized! Please provide a valid access token within the header [Authorization: Bearer <accessToken>] of your request.", "code": "COMMON_UNAUTHORIZED"}"

In [28]:
print(config.auth_token())

AttributeError: 'SHConfig' object has no attribute 'auth_token'

In [8]:
from oauthlib.oauth2 import BackendApplicationClient
from requests_oauthlib import OAuth2Session

# Your client credentials
client_id = 'sh-16f56f32-9a21-4587-99e7-3ccc42527d71'
client_secret = 'JeS1lvmqH5B1kZTnE9AiBRKs138nkOWs'

# Create a session
client = BackendApplicationClient(client_id=client_id)
oauth = OAuth2Session(client=client)

# Get token for the session
token = oauth.fetch_token(token_url='https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token',
                          client_secret=client_secret, include_client_id=True)

# All requests using this session will have an access token automatically added
resp = oauth.get("https://sh.dataspace.copernicus.eu/configuration/v1/wms/instances")
print(resp.content)

b'<html>\n<head>\n<meta http-equiv="Content-Type" content="text/html;charset=utf-8"/>\n<title>Error 500 Request failed.</title>\n</head>\n<body><h2>HTTP ERROR 500 Request failed.</h2>\n<table>\n<tr><th>URI:</th><td>/configuration/v1/wms/instances</td></tr>\n<tr><th>STATUS:</th><td>500</td></tr>\n<tr><th>MESSAGE:</th><td>Request failed.</td></tr>\n<tr><th>SERVLET:</th><td>org.glassfish.jersey.servlet.ServletContainer-14239223</td></tr>\n</table>\n\n</body>\n</html>\n'


In [11]:
for start, end in months:
    request = SentinelHubRequest(
        evalscript=evalscript,
        input_data=[
            SentinelHubRequest.input_data(
                data_collection=DataCollection.SENTINEL2_L2A,
                time_interval=(start, end),
                mosaicking_order="leastCC"  # dôležité
            )
        ],
        responses=[
            SentinelHubRequest.output_response(
                "default", MimeType.TIFF
            )
        ],
        bbox=bbox,
        size=(512, 512),
        config=config
    )

    data = request.get_data()

    filename = f"ndvi_{start[:7]}.tif"
    with open(filename, "wb") as f:
        f.write(data[0])

    print(f"Saved {filename}")

AttributeError: 'OAuth2Session' object has no attribute 'sh_base_url'